# EDA: Air Raid Alerts (Ukraine)

Exploratory analysis of raw alert events and hourly aggregated time series produced by the project pipeline.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from src.config import get_config
from src.main import run_pipeline

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

config = get_config()
raw_path = PROJECT_ROOT / config.raw_data_path
processed_path = PROJECT_ROOT / config.processed_data_path

In [ ]:
raw_df = pd.read_csv(raw_path, parse_dates=[config.datetime_column])
print(f"Raw rows: {len(raw_df)}")
print(f"Date range: {raw_df[config.datetime_column].min()} -> {raw_df[config.datetime_column].max()}")
raw_df.head()

In [ ]:
region_counts = raw_df[config.region_column].value_counts().sort_values(ascending=False)
region_counts

In [ ]:
processed_df, featured_df, regional_df = run_pipeline(config=config)
processed_df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
processed_df["alert_count"].plot(ax=ax, marker="o")
ax.set_title("Hourly Alert Counts")
ax.set_xlabel("Timestamp")
ax.set_ylabel("Alerts per hour")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
region_counts.plot(kind="bar", ax=ax)
ax.set_title("Alerts by Region (Raw Events)")
ax.set_xlabel("Region")
ax.set_ylabel("Event count")
plt.tight_layout()
plt.show()

## Baseline Models

Compare seasonal naive and linear regression baselines on a chronological holdout split.

In [ ]:
from src.main import run_baseline_evaluation

metrics_path = PROJECT_ROOT / config.baseline_metrics_path
summary_path = PROJECT_ROOT / config.summary_report_path
baseline_metrics = run_baseline_evaluation(
    processed_df=processed_df,
    featured_df=featured_df,
    regional_df=regional_df,
    config=config,
)
baseline_metrics["global"]

In [ ]:
global_metrics_df = pd.DataFrame(baseline_metrics["global"]).T
global_metrics_df

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
global_metrics_df[["mae", "rmse"]].plot(kind="bar", ax=ax)
ax.set_title("Global Baseline Model Comparison")
ax.set_xlabel("Model")
ax.set_ylabel("Error")
plt.tight_layout()
plt.show()

print(f"JSON report: {metrics_path}")
print(f"Markdown summary: {summary_path}")

## Regional Model Dashboard

Compare regional holdout errors across seasonal naive and LightGBM models.

In [ ]:
regional_rows = []
for region_name, models in baseline_metrics["regional"].items():
    for model_name, model_metrics in models.items():
        regional_rows.append(
            {
                "region": region_name,
                "model": model_name,
                "mae": model_metrics["mae"],
                "rmse": model_metrics["rmse"],
                "mape": model_metrics["mape"],
            }
        )

regional_metrics_df = pd.DataFrame(regional_rows)
regional_metrics_df

In [ ]:
pivot_mae = regional_metrics_df.pivot(index="region", columns="model", values="mae")
fig, ax = plt.subplots(figsize=(10, 5))
pivot_mae.plot(kind="bar", ax=ax)
ax.set_title("Regional MAE by Model")
ax.set_xlabel("Region")
ax.set_ylabel("MAE")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()